# Fine-tune riêng ĐÁNH – VẬT/KẸP CỔ – NGÃ

Ba sigmoid độc lập: `strike`, `grapple`, `fall`. Có thể đồng thời vật và ngã.
Bình thường là cả ba nhãn 0; không có một nhãn cảnh báo chung thay thế ba nhãn.
Giữ nguyên checkpoint cũ. Fine-tune ba đầu từ checkpoint cha; backbone AVA
và detector đóng băng. Đây chưa phải fine-tune toàn bộ backbone.

**Chạy kiểm tra nhãn trên CPU trước. Chỉ đổi sang GPU khi audit báo ready.**
Dữ liệu nhị phân cũ không tự đổi thành nhãn ba hành vi. `null` là chưa biết,
không phải âm. Clip đã dùng để test, gồm AIRTLab 37/87, phải giữ ở test.
Cần nhãn từng cửa sổ và video public trên Drive; không upload video iPhone.
GPU tối đa 30 phút cho trích đặc trưng và train, có dừng sớm validation.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, shutil, subprocess, sys, time, urllib.request, zipfile, zlib
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/SENTINEL')
DATA = DRIVE/'datasets'
MANIFEST = DATA/'three_actions_v1/review.json'
RUN_TAG = 'three_actions_'+time.strftime('%Y%m%d_%H%M%S')
RUN = DRIVE/'artifacts/three_actions_v1'/RUN_TAG
RUN.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/three_actions_v1'); WORK.mkdir(exist_ok=True)
CODE = WORK/'code'; DEMO = CODE/'mmaction2_demo'
DEMO.mkdir(parents=True, exist_ok=True)
WEIGHTS = WORK/'weights'; WEIGHTS.mkdir(exist_ok=True)
print('Manifest:', MANIFEST)
print('Output:', RUN)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import http.client
from google.colab import files

BUNDLE_SHA = 'c470ac5f7ddbc96be3f426edfcfca9b318af516d4eab2e85e8de5dc581d45d52'
BUNDLE = DRIVE/'artifacts/three_actions_v1'/('source_'+BUNDLE_SHA[:12]+'.zip')
if not BUNDLE.is_file():
    uploaded = files.upload()
    payload = next((value for value in uploaded.values() if hashlib.sha256(value).hexdigest() == BUNDLE_SHA), None)
    if payload is None:
        raise ValueError('Upload the verified three_actions_bundle.zip, not video or checkpoint files')
    BUNDLE.parent.mkdir(parents=True, exist_ok=True)
    BUNDLE.write_bytes(payload)
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != BUNDLE_SHA:
    raise ValueError('Source bundle checksum mismatch')
with zipfile.ZipFile(BUNDLE) as package:
    assert set(package.namelist()) == {'reviewed_manifest.json','visual_review.json','train_three_actions.py','action_engine.py','person_recovery.py'}
    overlays = {name: package.read(name).decode('utf-8') for name in ['train_three_actions.py','action_engine.py','person_recovery.py']}
    reviewed = json.loads(package.read('reviewed_manifest.json'))
for name, source in overlays.items():
    (DEMO/name).write_text(source,encoding='utf-8')
MANIFEST = DATA/'three_actions_v1/reviewed_20261110_v1.json'
MANIFEST.parent.mkdir(parents=True,exist_ok=True)
encoded = json.dumps(reviewed,ensure_ascii=False,indent=2)
if MANIFEST.exists() and json.loads(MANIFEST.read_text()) != reviewed:
    raise ValueError('Existing review manifest changed; use a new review version')
MANIFEST.write_text(encoded,encoding='utf-8')

def public_request(url):
    if not url.startswith(('https://raw.githubusercontent.com/','https://api.github.com/repos/')):
        raise ValueError('Unexpected public download host')
    for attempt in range(3):
        try:
            request = urllib.request.Request(url,headers={'User-Agent':'three-action-reviewed-colab'})
            with urllib.request.urlopen(request,timeout=90) as response:
                return response.read(26*1024*1024)
        except (OSError,http.client.HTTPException):
            if attempt==2:
                raise
            time.sleep(attempt+1)

def prepare_source(item):
    target=(DATA/item['video']).resolve()
    assert target.is_relative_to(DATA.resolve())
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest()==item['sha256']:
        return
    try:
        content=public_request(item['download_url'])
    except (OSError,http.client.HTTPException):
        content=base64.b64decode(json.loads(public_request(item['blob_url']))['content'])
    assert len(content)==item['size'] and hashlib.sha256(content).hexdigest()==item['sha256'],item['video']
    temporary=target.with_suffix('.partial')
    temporary.write_bytes(content)
    temporary.replace(target)

with ThreadPoolExecutor(max_workers=3) as pool:
    for number,_ in enumerate(pool.map(prepare_source,reviewed['sources']),1):
        if number%5==0 or number==len(reviewed['sources']):
            print('PUBLIC DATA VERIFIED',number,'/',len(reviewed['sources']),flush=True)
subprocess.run([sys.executable,str(DEMO/'train_three_actions.py'),'--manifest',str(MANIFEST),
    '--root',str(DATA),'--output',str(RUN),'--stage','audit'],check=True)
audit=json.loads((RUN/'label_audit.json').read_text())
assert audit['status']=='ready',audit['issues']
print('AUDIT READY:',len(reviewed['sources']),'public clips,',audit['windows'],'reviewed windows; no private iPhone data')


## Chỉ chạy phần sau khi audit ready và đã chọn GPU
Không bỏ qua kiểm tra nhãn. Đường dẫn video trong review.json tính từ SENTINEL/datasets.
Dùng group theo người/phiên quay, không chia ngẫu nhiên các cửa sổ cùng video.


In [ ]:
audit = json.loads((RUN/'label_audit.json').read_text())
assert audit['status'] == 'ready', 'Không train trên nhãn gộp hoặc chưa duyệt'
subprocess.run(['nvidia-smi'],check=True)
SOURCE_REVISION = 'fda444194032f69845176887eee9b978d3373876'
archive = WORK/'source.zip'
with urllib.request.urlopen('https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/'+SOURCE_REVISION,timeout=180) as response, archive.open('wb') as output:
    shutil.copyfileobj(response,output)
with zipfile.ZipFile(archive) as package:
    assert all((WORK/item.filename).resolve().is_relative_to(WORK.resolve()) for item in package.infolist())
    package.extractall(WORK)
source = next(path for path in WORK.iterdir() if (path/'mmaction2_demo/train_fixed_running.py').is_file())
shutil.copytree(source,CODE,dirs_exist_ok=True)
for name, source in overlays.items():
    (DEMO/name).write_text(source,encoding='utf-8')


In [ ]:
# Mọi checkpoint đầu vào đều lấy từ Google Drive; không dùng upload thủ công.
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth

checks = {
    'interaction_fall_wrestle.pt': ('1K_KOzEyxpbrnQExEcS4Hy3UkeMCn144_',
      'ddb09180284c4093189c53b7decbfcc5b51fd2d9b7700536d104524bceb5c344'),
    'interaction_fall_wrestle.json': ('1Via7ET9g6iQMjR-pX7NRpj9gKQktFQWe', None),
    'action_slowonly_r101.pth': ('1aU7C2tCp6NQc1NDm5GRwdkWtgqkr3lkt',
      '163785949c5178484012a6593d12d6fcf4d36be404fa42b5899b314e91f4703b'),
    'person_faster_rcnn_r50.pth': ('1gbBBgvKhp1p3FNDRoTsV92_t2vqMwUyP',
      'a5d8aa15c75b961b9c73f28a6e92af370835639a2939ff890852b052a5256a5b'),
}
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
service = None
def drive_download(file_id, target):
    global service
    if service is None:
        auth.authenticate_user()
        credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
        service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    request = service.files().get_media(fileId=file_id)
    temporary = target.with_suffix(target.suffix+'.partial')
    with temporary.open('wb') as output:
        downloader = MediaIoBaseDownload(output, request, chunksize=8*1024*1024)
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=3)
    temporary.replace(target)
parent_run = DRIVE/'artifacts/mmaction_interaction_fall_wrestle/fall_wrestle_pilot_01'
parent_public = DRIVE/'artifacts/mmaction_interaction_fall_wrestle/public_weights'
alternatives = {
    'interaction_fall_wrestle.pt': [parent_run/'interaction.pt'],
    'interaction_fall_wrestle.json': [parent_run/'interaction.json'],
    'action_slowonly_r101.pth': [parent_public/'action_slowonly_r101.pth'],
    'person_faster_rcnn_r50.pth': [parent_public/'person_faster_rcnn_r50.pth'],
}
for name, (file_id, expected) in checks.items():
    target = WEIGHTS/name
    if target.is_file() and (expected is None or sha256(target) == expected):
        continue
    source = next((p for p in alternatives[name]
                   if p.is_file() and (expected is None or sha256(p) == expected)), None)
    if source is not None:
        shutil.copyfile(source, target)
    else:
        drive_download(file_id, target)
    if expected is not None and sha256(target) != expected:
        raise ValueError('Sai SHA-256 checkpoint Drive: '+name)
meta = json.loads((WEIGHTS/'interaction_fall_wrestle.json').read_text(encoding='utf-8'))
assert meta['checkpoint_sha256'] == checks['interaction_fall_wrestle.pt'][1]
print('Đã xác minh checkpoint cha và 2 trọng số gốc từ Drive')


In [ ]:
# Môi trường giống pipeline MMAction2 đã dùng để train checkpoint cha.
os.environ.update(MPLBACKEND='Agg', CUDA_MODULE_LOADING='LAZY', OMP_NUM_THREADS='4')
ENV = WORK/'env'; PY = ENV/'bin/python'
def run(command, label, timeout=1800):
    log = RUN/(label+'.log')
    with log.open('a', encoding='utf-8') as output:
        result = subprocess.run([str(x) for x in command], stdout=output,
                                stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        raise RuntimeError(f'{label} failed; see {log}\n{log.read_text(encoding="utf-8")[-4000:]}')
    print(label, 'OK')
if not PY.is_file():
    run([sys.executable,'-m','pip','install','-q','uv==0.8.22'], 'install_uv')
    run(['uv','python','install','3.10'], 'install_python')
    run(['uv','venv','--python','3.10','--seed',ENV], 'create_env')
run([PY,'-m','pip','install','-q','numpy==1.26.4','setuptools==69.5.1','wheel'], 'install_base')
run([PY,'-m','pip','install','-q','torch==2.0.1+cu118','torchvision==0.15.2+cu118',
     '--index-url','https://download.pytorch.org/whl/cu118'], 'install_torch')
run([PY,'-m','pip','install','-q','mmcv==2.1.0','--only-binary=mmcv',
     '-f','https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html',
     'mmengine==0.10.7','mmdet==3.2.0','decord==0.6.0','einops==0.8.1',
     'importlib-metadata','scikit-learn==1.3.2','opencv-python==4.8.1.78'], 'install_openmmlab')
if not (DEMO/'repo/.git').is_dir():
    run(['git','clone','--depth','1','--branch','v1.2.0',
         'https://github.com/open-mmlab/mmaction2.git',DEMO/'repo'], 'clone_mmaction')
run([PY,'-m','pip','install','--no-deps','--no-build-isolation','-e',DEMO/'repo'], 'install_mmaction')
os.environ['PYTHONPATH'] = os.pathsep.join(map(str,(CODE,DEMO,DEMO/'repo')))
run([PY,'-c','import torch, mmcv; from mmcv.ops import roi_align; assert torch.cuda.is_available()'], 'check_cuda')


In [ ]:
os.environ['MPLBACKEND']='Agg'
command=[PY,DEMO/'train_three_actions.py','--manifest',MANIFEST,'--root',DATA,
    '--weights',WEIGHTS,'--output',RUN,'--stage','train','--minutes','30','--epochs','120']
run(command,'train_three_actions',timeout=2100)
report=json.loads((RUN/'report.json').read_text())
assert report['status']=='completed'
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Checkpoint riêng, chưa cài lên web:',RUN/'three_actions_pilot.zip')
